# 2 Stage Stochastic Program: Inventory stocking with Scenarios 

## Choose the best inventory stocking decision with respect to uncertain future demand. 

###  Bus 36109 "Advanced Decision Modeling with Python", Don Eisenstein
Don Eisenstein &copy; Copyright 2021, University of Chicago 

---

We need to choose a weekly stocking level `X` in anticipation of uncertain weekly demand.

For each unit short for the week a cost of `C_s` is incurred.  For each unit stocked in excess of demand a cost of `C_e` is incurred.  

We have collected historical weekly demand data with the following equally likely 10 scenarios:


| Scenario  | weekly demand | 
|--|:--:|
| **0** | 230 | 
| **1** | 210 |
| **2** | 160 |
| **3** | 450 |
| **4** | 310 |
| **5** | 190 |
| **6** | 130 |
| **7** | 340 |
| **8** | 370 |
| **9** | 290 |

What stocking decision `X` should be made to minimize the expected cost?

### Linear Programming Formulation

- First Stage Decisions Vars
  - Let X be the weekly stocking decision 


- Second Stage Decisions Vars
  - Let S_i be the number of units short for scenario i 
  - Let E_i be the number of units in excess for scenario i 


- Minimize  0.10 (C_s S_0 + C_e E_0) \
\+ 0.10 (C_s S_1 + C_e E_1) \
\+ 0.10 (C_s S_2 + C_e E_2) \
\+ 0.10 (C_s S_3 + C_e E_3) \
\+ 0.10 (C_s S_4 + C_e E_4) \
\+ 0.10 (C_s S_5 + C_e E_5) \
\+ 0.10 (C_s S_6 + C_e E_6) \
\+ 0.10 (C_s S_7 + C_e E_7) \
\+ 0.10 (C_s S_8 + C_e E_8) \
\+ 0.10 (C_s S_9 + C_e E_9) 


- First Stage: 
  - X >= 0
  
  
- Second Stage:
    - X + S_0 - E_0 = 230 
    - X + S_1 - E_1 = 210 
    - X + S_2 - E_2 = 160 
    - X + S_3 - E_3 = 450 
    - X + S_4 - E_4 = 310 
    - X + S_5 - E_5 = 190 
    - X + S_6 - E_6 = 130 
    - X + S_7 - E_7 = 340 
    - X + S_8 - E_8 = 370 
    - X + S_9 - E_9 = 290 
    

In [1]:
demand = [ 230, 210, 160, 450, 310, 190, 130, 340, 370, 290]
C_s = 20
C_e = 10
mean_demand = sum(demand)/len(demand)
print(f"Average demand: {mean_demand}")

Average demand: 268.0


In [2]:
# === SETUP ===
from pulp import *
import os
from pprint import pprint

# Portable solver setup, to allow work locally (ARM64 architecture) as well as in JupyterHub
from pulp import COIN_CMD
if os.path.exists('/opt/homebrew/opt/cbc/bin/cbc'):
    solver = COIN_CMD(path='/opt/homebrew/opt/cbc/bin/cbc', msg=0)
else:
    solver = pulp.PULP_CBC_CMD(msg=0)

In [3]:
model = LpProblem("Inventory_Planning",LpMinimize)

#### Stage 1 variable(s)

In [4]:
X = pulp.LpVariable('X', lowBound=0, cat='Continuous')

#### Stage 2 variable(s)

In [5]:
S = [] # Inventory shortages
E = [] # Inventory excesses
num_scenarios = len(demand)

for num in range(num_scenarios):
    S.append(pulp.LpVariable(f"S_{num}", lowBound=0, cat='Continuous'))
    E.append(pulp.LpVariable(f"E_{num}", lowBound=0, cat='Continuous'))
print(S)
print(E)

[S_0, S_1, S_2, S_3, S_4, S_5, S_6, S_7, S_8, S_9]
[E_0, E_1, E_2, E_3, E_4, E_5, E_6, E_7, E_8, E_9]


#### Objective function

In [6]:
obj = None
for num in range(num_scenarios):
    obj += 1.0/num_scenarios * (C_e * E[num] + C_s * S[num])
model += obj, "obj"
print(model)

Inventory_Planning:
MINIMIZE
1.0*E_0 + 1.0*E_1 + 1.0*E_2 + 1.0*E_3 + 1.0*E_4 + 1.0*E_5 + 1.0*E_6 + 1.0*E_7 + 1.0*E_8 + 1.0*E_9 + 2.0*S_0 + 2.0*S_1 + 2.0*S_2 + 2.0*S_3 + 2.0*S_4 + 2.0*S_5 + 2.0*S_6 + 2.0*S_7 + 2.0*S_8 + 2.0*S_9 + 0.0
VARIABLES
E_0 Continuous
E_1 Continuous
E_2 Continuous
E_3 Continuous
E_4 Continuous
E_5 Continuous
E_6 Continuous
E_7 Continuous
E_8 Continuous
E_9 Continuous
S_0 Continuous
S_1 Continuous
S_2 Continuous
S_3 Continuous
S_4 Continuous
S_5 Continuous
S_6 Continuous
S_7 Continuous
S_8 Continuous
S_9 Continuous



#### Constraint(s): inventory balance

In [7]:
for num in range(num_scenarios):
    model += X + S[num] - E[num] == demand[num], f"Inventory Balance {num}"
print(model)

Inventory_Planning:
MINIMIZE
1.0*E_0 + 1.0*E_1 + 1.0*E_2 + 1.0*E_3 + 1.0*E_4 + 1.0*E_5 + 1.0*E_6 + 1.0*E_7 + 1.0*E_8 + 1.0*E_9 + 2.0*S_0 + 2.0*S_1 + 2.0*S_2 + 2.0*S_3 + 2.0*S_4 + 2.0*S_5 + 2.0*S_6 + 2.0*S_7 + 2.0*S_8 + 2.0*S_9 + 0.0
SUBJECT TO
Inventory_Balance_0: - E_0 + S_0 + X = 230

Inventory_Balance_1: - E_1 + S_1 + X = 210

Inventory_Balance_2: - E_2 + S_2 + X = 160

Inventory_Balance_3: - E_3 + S_3 + X = 450

Inventory_Balance_4: - E_4 + S_4 + X = 310

Inventory_Balance_5: - E_5 + S_5 + X = 190

Inventory_Balance_6: - E_6 + S_6 + X = 130

Inventory_Balance_7: - E_7 + S_7 + X = 340

Inventory_Balance_8: - E_8 + S_8 + X = 370

Inventory_Balance_9: - E_9 + S_9 + X = 290

VARIABLES
E_0 Continuous
E_1 Continuous
E_2 Continuous
E_3 Continuous
E_4 Continuous
E_5 Continuous
E_6 Continuous
E_7 Continuous
E_8 Continuous
E_9 Continuous
S_0 Continuous
S_1 Continuous
S_2 Continuous
S_3 Continuous
S_4 Continuous
S_5 Continuous
S_6 Continuous
S_7 Continuous
S_8 Continuous
S_9 Continuous
X Cont

In [9]:
model.solve(solver)
print(f"Status: {LpStatus[model.status]}")

Status: Optimal


In [ ]:
for v in model.variables():
    print(f"{v.name} = {v.varValue}")

### The Newsvendor Model (a side note)

In this simple case we can solve the problem using a model known as the Newsvendor.  This is a model covered in all the core Operations classes.

For those already familiar let's check our answer:

Here, we note that the optimal service level is `SL = Cs/(Cs+Ce) = 20/(20+10) = 0.667`

The demands sorted are `[130, 160, 190, 210, 230, 290, 310, 340, 370, 450]`

So finding `0.667` into the cumulative distribution puts us between `290` and `310`, and you might recall in a discrete distribution we always round up ... so the optimal stocking level is indeed `X=310`.

The advantage to our optimization appraoch is we can handle much more complex problems that the simple Newsvendor can not, but this was an easy place for us to start and verify our work.

We note that setting the stocking decision at the mean demand of 268 is less than our optimal. 
 